This is the code for downloading the dataset for Prjoect 1

In [ ]:
# --- Imports & Setup ---
import wrds
import pandas as pd
import os
import numpy as np

# --- Connect to WRDS ---
wrds_db = wrds.Connection()

# --- Download CRSP monthly returns ---
crsp = wrds_db.raw_sql("""
    SELECT permno, date, ret, shrout, prc, altprc, vol, retx
    FROM crsp.msf
    WHERE date >= '1990-01-01'
""")

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['yyyymm'] = crsp['date'].dt.year * 100 + crsp['date'].dt.month
crsp['me'] = crsp['prc'].abs() * crsp['shrout']
crsp['logme'] = np.log(crsp['me'])

# --- Load predictive characteristics (from jkpfactors.com) ---
chars = pd.read_csv("signed_predictors_dl_wide.csv")

# --- Merge datasets ---
merged = pd.merge(chars, crsp, on=['permno', 'yyyymm'], how='inner')
merged = merged.fillna(0)
merged.columns = merged.columns.str.lower()


# --- Standardize features cross-sectionally by month ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
feature_cols = [col for col in merged.columns if col not in exclude_cols]
merged[feature_cols] = merged.groupby('yyyymm')[feature_cols].transform(
    lambda x: (x - x.mean()) / x.std()
)
merged = merged.fillna(0)

# --- Save yearly .parquet files ---
os.makedirs("dataset_yearly_parquet", exist_ok=True)
merged['year'] = merged['yyyymm'] // 100

for year in sorted(merged['year'].unique()):
    chunk = merged[merged['year'] == year]
    filename = f"dataset_yearly_parquet/prepared_{year}.parquet"
    chunk.to_parquet(filename, index=False)
    print(f"Saved {filename} with shape {chunk.shape}")


# **If you have already downloaded the yearly parquet files from GitHub, ONLY RUN THIS BLOCK
These are some diagnostic tests to ensure you data is working properly.

In [ ]:
# --- Imports (if needed) ---
import pandas as pd
import glob
from sklearn.linear_model import LinearRegression
import numpy as np

# --- Load full dataset ---
all_files = glob.glob("dataset_yearly_parquet/*.parquet")
df = pd.concat([pd.read_parquet(f) for f in all_files])
print("Full dataset shape:", df.shape)

# --- Define features ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# --- Quick check: standardization across time ---
sample_cols = feature_cols[:5]
print(df.groupby('yyyymm')[sample_cols].agg(['mean', 'std']).head())

# --- Replicate 3-factor OLS model ---
factors = ['logme', 'bm', 'mom12m']
assert all(f in df.columns for f in factors), "Missing one of the 3 benchmark factors"

X = df[factors]
y = df['ret']
mask = X.notnull().all(axis=1) & y.notnull()
X = X[mask]
y = y[mask]

model = LinearRegression()
model.fit(X, y)
r2 = model.score(X, y)

print(f"Benchmark OLS (3-factor) R²: {r2 * 100:.4f}%")

# --- Duplication check ---
dupes = df.duplicated(subset=['permno', 'yyyymm'])
print(f"Duplicate rows by permno + yyyymm: {dupes.sum()}")

In [ ]:
#This prints all of the variable names in the dataset
import pprint
pprint.pprint(sorted(df.columns.tolist()))